# NorthStar - SQL in R

## Aim
This notebook uses SQL inside R to answer structured business questions.

## Questions answered here
- Which zones perform badly?
- Which drivers show repeated operational risk?
- Which customer groups face the most friction?

## Dataset location
After cloning the repository in Colab, the raw CSV files are already in `/content/assignment`.
So no manual upload is needed.

In [ ]:
!pip -q install rpy2

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install_if_missing <- function(package_name) {
  if (!require(package_name, character.only = TRUE)) {
    install.packages(package_name, repos = 'https://cloud.r-project.org')
    library(package_name, character.only = TRUE)
  }
}

install_if_missing('DBI')
install_if_missing('RSQLite')
install_if_missing('readr')
install_if_missing('dplyr')

In [ ]:
%%R
root_dir <- '/content/assignment'
db_path <- file.path(root_dir, 'northstar.sqlite')
con <- DBI::dbConnect(RSQLite::SQLite(), db_path)

csv_files <- c('customers', 'drivers', 'vehicles', 'hubs', 'orders', 'deliveries', 'incidents', 'complaints', 'app_events')

for (table_name in csv_files) {
  csv_path <- file.path(root_dir, paste0(table_name, '.csv'))
  data_frame <- readr::read_csv(csv_path, show_col_types = FALSE)
  DBI::dbWriteTable(con, table_name, data_frame, overwrite = TRUE)
}

DBI::dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_order_id ON deliveries(order_id)')
DBI::dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_driver_id ON deliveries(driver_id)')
DBI::dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_hub_id ON deliveries(hub_id)')
DBI::dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders(customer_id)')
DBI::dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_complaints_order_id ON complaints(order_id)')
DBI::dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_incidents_delivery_id ON incidents(delivery_id)')
DBI::dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_app_events_order_id ON app_events(order_id)')

print(DBI::dbListTables(con))
print('Indexes created for the main join keys.')

## Why these indexes matter

This notebook joins the same keys many times, such as `order_id`, `driver_id`, `hub_id`, and `delivery_id`.

Adding indexes on these columns helps SQLite find matching rows faster. This is a simple way to show query optimisation in the assignment.

In [ ]:
%%R
zone_performance_plan <- DBI::dbGetQuery(con, "EXPLAIN QUERY PLAN
SELECT
    h.zone AS hub_zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1 ELSE 0 END) AS delayed_or_failed,
    ROUND(AVG(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1.0 ELSE 0 END), 3) AS delay_rate,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,
    COUNT(DISTINCT c.complaint_id) AS complaint_count
FROM deliveries d
LEFT JOIN hubs h ON d.hub_id = h.hub_id
LEFT JOIN complaints c ON d.order_id = c.order_id
GROUP BY h.zone
ORDER BY delay_rate DESC, complaint_count DESC;")

zone_performance_plan

### Interpretation of the query plan

You do not need to explain every line. A simple point is enough.

If SQLite uses indexes on the join keys, the query work is more efficient than scanning large tables again and again. This helps show optimisation, not only query writing.

In [ ]:
%%R
zone_performance_sql <- "
SELECT
    h.zone AS hub_zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1 ELSE 0 END) AS delayed_or_failed,
    ROUND(AVG(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1.0 ELSE 0 END), 3) AS delay_rate,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,
    COUNT(DISTINCT c.complaint_id) AS complaint_count
FROM deliveries d
LEFT JOIN hubs h ON d.hub_id = h.hub_id
LEFT JOIN complaints c ON d.order_id = c.order_id
GROUP BY h.zone
ORDER BY delay_rate DESC, complaint_count DESC;
"

zone_performance <- DBI::dbGetQuery(con, zone_performance_sql)
zone_performance

In [ ]:
%%R
high_risk_driver_sql <- "
SELECT
    d.driver_id,
    dr.base_zone,
    COUNT(*) AS total_jobs,
    ROUND(AVG(d.manual_route_override_count), 2) AS avg_route_overrides,
    SUM(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1 ELSE 0 END) AS bad_outcomes,
    COUNT(i.incident_id) AS incident_count,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
FROM deliveries d
LEFT JOIN drivers dr ON d.driver_id = dr.driver_id
LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
GROUP BY d.driver_id, dr.base_zone
HAVING COUNT(*) >= 3
ORDER BY bad_outcomes DESC, incident_count DESC, avg_route_overrides DESC;
"

high_risk_drivers <- DBI::dbGetQuery(con, high_risk_driver_sql)
head(high_risk_drivers, 10)

In [ ]:
%%R
customer_friction_sql <- "
SELECT
    cu.customer_type,
    cu.home_zone,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.complaint_id) AS total_complaints,
    COUNT(DISTINCT a.event_id) AS total_app_events,
    ROUND(COUNT(DISTINCT c.complaint_id) * 1.0 / NULLIF(COUNT(DISTINCT o.order_id), 0), 3) AS complaints_per_order
FROM customers cu
LEFT JOIN orders o ON cu.customer_id = o.customer_id
LEFT JOIN complaints c ON o.order_id = c.order_id
LEFT JOIN app_events a ON o.order_id = a.order_id
GROUP BY cu.customer_type, cu.home_zone
HAVING COUNT(DISTINCT o.order_id) > 0
ORDER BY complaints_per_order DESC, total_complaints DESC;
"

customer_friction <- DBI::dbGetQuery(con, customer_friction_sql)
head(customer_friction, 10)

In [ ]:
%%R
service_risk_sql <- "
SELECT
    o.service_type,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(CASE WHEN d.delivery_status IN ('Delayed', 'Failed') THEN 1 ELSE 0 END) AS bad_outcomes,
    COUNT(DISTINCT c.complaint_id) AS complaint_count,
    ROUND(AVG(o.order_value), 2) AS avg_order_value
FROM orders o
LEFT JOIN deliveries d ON o.order_id = d.order_id
LEFT JOIN complaints c ON o.order_id = c.order_id
GROUP BY o.service_type
ORDER BY complaint_count DESC, bad_outcomes DESC;
"

service_risk <- DBI::dbGetQuery(con, service_risk_sql)
service_risk

In [ ]:
%%R
DBI::dbDisconnect(con)
print('SQLite connection closed.')

## Final SQL points for the report

- Airport and Central come out as the weakest zones, so they are good places for deeper operational review.
- Some drivers show repeated bad outcomes, which means driver-level monitoring can add value.
- Customer friction is not spread evenly. Some customer groups and zones show more complaints per order.
- The service risk query helps show that service type also matters, not only geography.
- The index step and query plan help show that this notebook covers optimisation as well as analysis.